# Ejercicio 8 - Evolucion 2024, 2025 y 2026

Este notebook grafica los resultados del Ejercicio 8. No ejecuta consultas pesadas: lee los CSV que generan

- `python scripts/run_sql.py ejercicio7 --salida ejercicio8/indicadores_3_anios --tablero` (los 12 indicadores del Ejercicio 7 sobre los tres anios) y
- `python scripts/run_sql.py ejercicio8 --tablero` (consultas de evolucion 8.1 a 8.5),

y, si falta alguno, ejecuta la consulta en DuckDB sobre las vistas (todos los anios descargados).

Documentacion e interpretacion: `docs/ejercicio8_2025.md`. Figuras en `docs/figuras/e8_*.png`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))   # notebooks/ -> scripts/
from lab import conectar, ejecutar_sql, RAIZ_REPO

import matplotlib.pyplot as plt
import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DIR_FIG = RAIZ_REPO / "docs" / "figuras"
DIR_FIG.mkdir(parents=True, exist_ok=True)
# Paleta categorica validada (la misma del Ejercicio 5 y del tablero). El color
# sigue a la entidad: cada anio conserva su color en todas las figuras.
AZUL, NARANJA, AGUA, GRIS = "#2a78d6", "#eb6834", "#1baf7a", "#8a8f98"
COLOR_ANIO = {2024: AZUL, 2025: AGUA, 2026: NARANJA}
TINTA, TINTA_2, REJILLA = "#0b0b0b", "#52514e", "#e6e5e0"
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": TINTA_2, "axes.labelcolor": TINTA_2, "xtick.color": TINTA_2,
    "ytick.color": TINTA_2, "axes.grid": True, "grid.color": REJILLA, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "lines.linewidth": 2, "lines.markersize": 6,
    "legend.frameon": False,
})

def guardar(fig, nombre):
    fig.tight_layout()
    fig.savefig(DIR_FIG / f"{nombre}.png", bbox_inches="tight")
    plt.show()

DIR_RES = RAIZ_REPO / "docs" / "resultados"
con = None

def cargar(archivo, carpeta):
    """Lee el CSV generado por scripts/run_sql.py; si no existe, ejecuta la consulta."""
    global con
    csv = DIR_RES / carpeta / (Path(archivo).stem + ".csv")
    if csv.exists():
        print(f"[{Path(archivo).stem}] leido de {csv.relative_to(RAIZ_REPO)}")
        return pd.read_csv(csv)
    print(f"[{Path(archivo).stem}] no hay CSV: se ejecuta la consulta")
    con = con or conectar()
    return ejecutar_sql(con, archivo)[1]

IND = "ejercicio8/indicadores_3_anios"   # indicadores del Ejercicio 7 con 2024-2026

## 8.4 - 8.5 Demanda: viajes por dia en cada mes, tres anios

Indicador I1. Un panel por tipo de taxi porque su escala es unas 90 veces distinta; el mes en el eje x permite comparar el mismo mes de cada anio.

In [ ]:
d = cargar("sql/ejercicio7/7_01_demanda_diaria.sql", IND)
fig, ejes = plt.subplots(1, 2, figsize=(12, 3.9))
for eje, taxi in zip(ejes, ["yellow", "green"]):
    t = d[d.taxi == taxi]
    for anio, color in COLOR_ANIO.items():
        s = t[t.anio == anio].sort_values("mes")
        if s.empty:
            continue
        eje.plot(s.mes, s.viajes_por_dia, marker="o", color=color, label=str(anio),
                 markeredgecolor="white", markeredgewidth=1.5)
        ultimo = s.iloc[-1]
        eje.annotate(str(anio), (ultimo.mes, ultimo.viajes_por_dia), xytext=(6, 0),
                     textcoords="offset points", va="center", color=TINTA, fontsize=9)
    eje.set_xticks(range(1, 13), MESES)
    eje.set_xlim(0.6, 12.9)
    eje.set_ylim(bottom=0)
    eje.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
    eje.set_title(f"{taxi}: viajes validos por dia", color=TINTA, loc="left")
    eje.legend(loc="lower left", ncol=3)
guardar(fig, "e8_viajes_por_dia_3_anios")
d.pivot_table(index="mes", columns=["taxi", "anio"], values="viajes_por_dia")

## Resumen por anio en los meses comunes (I12)

In [ ]:
r = cargar("sql/ejercicio7/7_12_resumen_anual.sql", IND)
r.set_index(["taxi", "anio"]).T

## 8.6 Patron 1: velocidad dentro de la zona CBD frente al resto de Manhattan (consulta 8.1)

Viajes amarillos de lunes a viernes de 7 a 19 h. La linea vertical marca el inicio del cobro por congestion (5 de enero de 2025). Un solo eje: ambas series estan en mph.

In [ ]:
v = cargar("sql/ejercicio8/8_01_velocidad_cbd_vs_control.sql", "ejercicio8")
v["periodo"] = pd.to_datetime(v.periodo)
COLOR_GRUPO = {"Dentro de la zona CBD": NARANJA, "Manhattan fuera de la zona": AZUL}
fig, eje = plt.subplots(figsize=(12, 4))
for grupo, color in COLOR_GRUPO.items():
    s = v[v.grupo == grupo].sort_values("periodo")
    eje.plot(s.periodo, s.mph_mediana, marker="o", color=color, label=grupo,
             markeredgecolor="white", markeredgewidth=1.5)
eje.axvline(pd.Timestamp("2025-01-05"), color=TINTA_2, linewidth=1, linestyle="--")
eje.annotate("inicio del cobro CBD (5 ene 2025)", (pd.Timestamp("2025-01-05"), 1),
             xytext=(4, 0), textcoords="offset points", color=TINTA_2, fontsize=9)
eje.set_ylim(bottom=0)
eje.set_ylabel("mph (mediana)")
eje.set_title("Velocidad mediana de viajes amarillos, lun-vie 7-19 h", color=TINTA, loc="left")
eje.legend(loc="lower right")
guardar(fig, "e8_velocidad_cbd_vs_control")

# Promedio de las medianas mensuales en los meses comunes (enero a agosto)
ultimo_mes = v[v.anio == v.anio.max()].mes.max()
p = v[v.mes <= ultimo_mes].groupby(["grupo", "anio"]).mph_mediana.mean().unstack()
p.loc["cociente CBD / fuera"] = p.loc["Dentro de la zona CBD"] / p.loc["Manhattan fuera de la zona"]
p.round(3)

## 8.6 Patron 2: de donde viene el cambio de volumen (consulta 8.2)

Viajes amarillos por dia en los meses comunes, por franja horaria y forma de pago. Un panel por franja; dentro de cada panel, una barra apilada por anio.

In [ ]:
g = cargar("sql/ejercicio8/8_02_crecimiento_por_franja_y_pago.sql", "ejercicio8")
y = g[g.taxi == "yellow"]
franjas = sorted(y.franja.unique())
COLOR_PAGO = {"Tarjeta, efectivo y otros": AZUL, "Flex fare / sin dato": NARANJA}
fig, ejes = plt.subplots(1, len(franjas), figsize=(13, 4.2), sharey=True)
for eje, franja in zip(ejes, franjas):
    t = y[y.franja == franja].pivot(index="anio", columns="pago", values="viajes_por_dia")
    base = 0
    for pago, color in COLOR_PAGO.items():
        eje.bar(t.index.astype(str), t[pago], bottom=base, color=color, label=pago,
                edgecolor="white", linewidth=2, width=0.7)
        base = base + t[pago]
    for x, total in zip(t.index.astype(str), base):
        eje.annotate(f"{total:,.0f}", (x, total), xytext=(0, 3), textcoords="offset points",
                     ha="center", fontsize=8, color=TINTA)
    eje.set_title(franja[2:], color=TINTA, fontsize=10)
    eje.grid(axis="x", visible=False)
ejes[0].set_ylabel("viajes amarillos por dia")
ejes[0].yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
fig.legend(*ejes[0].get_legend_handles_labels(), loc="upper right", ncol=2, fontsize=9)
fig.suptitle("Viajes amarillos por dia por franja y forma de pago (enero a agosto)",
             x=0.01, ha="left", color=TINTA)
guardar(fig, "e8_crecimiento_franja_pago")
g.pivot_table(index=["taxi", "pago"], columns="anio", values="viajes_por_dia", aggfunc="sum")

## 8.6 Patron 3: registros con monto <= 0 por mes (I11 y consulta 8.3)

La regla de monto de `viajes_validos` excluye, de enero a noviembre de 2025, muchos mas registros amarillos que en 2024 o 2026.

In [ ]:
q = cargar("sql/ejercicio7/7_11_calidad.sql", IND)
q["periodo"] = pd.to_datetime(q.periodo)
fig, eje = plt.subplots(figsize=(12, 3.6))
for taxi, color in (("yellow", NARANJA), ("green", AZUL)):
    s = q[q.taxi == taxi].sort_values("periodo")
    eje.plot(s.periodo, s.pct_monto, marker="o", color=color, label=taxi,
             markeredgecolor="white", markeredgewidth=1.5)
eje.set_ylim(bottom=0)
eje.set_ylabel("% de los registros del mes")
eje.set_title("Registros con tarifa o total <= 0 (regla f_monto)", color=TINTA, loc="left")
eje.legend(loc="upper left")
guardar(fig, "e8_calidad_monto")

m = cargar("sql/ejercicio8/8_03_desglose_monto_cero.sql", "ejercicio8")
m["grupo"] = m.vendor_id.astype(str) + " " + m.metodo_pago
t = m.pivot_table(index=["anio", "mes"], columns="grupo", values="pct_de_registros_del_mes", aggfunc="sum")
t.loc[:, t.max() > 0.3].round(2)

## Precio: componentes del total y efecto de la forma de pago (consultas 8.4 y 8.5)

In [ ]:
comp = cargar("sql/ejercicio8/8_04_componentes_del_total.sql", "ejercicio8")
comp[comp.mes <= 8].groupby("anio").mean(numeric_only=True).drop(columns="mes").round(2)

In [ ]:
precio = cargar("sql/ejercicio8/8_05_precio_por_forma_de_pago.sql", "ejercicio8")
precio.pivot(index="pago", columns="anio", values=["pct_viajes", "total_mediano"])

## Otros indicadores en los tres anios

In [ ]:
pago = cargar("sql/ejercicio7/7_06_metodo_pago.sql", IND)
pago[pago.metodo == "Flex fare / sin dato"].pivot_table(index="mes", columns=["taxi", "anio"], values="pct")

In [ ]:
cbd = cargar("sql/ejercicio7/7_10_cargo_congestion.sql", IND)
cbd[cbd.anio >= 2025].pivot_table(index="mes", columns=["taxi", "anio"], values=["pct_con_cargo", "recaudacion_por_dia"])

In [ ]:
aer = cargar("sql/ejercicio7/7_08_aeropuertos.sql", IND)
aer[aer.aeropuerto != "Sin aeropuerto"].groupby(["taxi", "anio"])[["pct_viajes", "pct_facturacion"]].sum()

In [ ]:
prop = cargar("sql/ejercicio7/7_07_propina.sql", IND)
prop.groupby(["taxi", "anio"])[["pct_con_propina", "pct_propina_20", "propina_pct_mediana"]].mean().round(2)